<a href="https://colab.research.google.com/github/costpetrides/Numerical-Methods-GFD/blob/main/Lab_04_Heat_explicit_solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Εξίσωση διάδοσης θερμότητας

## Explicit FTCS
________

Αναζητούμε τη χρονική εξέλιξη της θερμοκρασίας σε μια ράβδο, όπου η θερμοκρασία στα άκρα της παραμένει σταθερή.

Επομένως, θέλουμε την αριθμητική λύση της εξίσωσης θερμότητας:

$$
\begin{equation*}
\frac{\partial u}{\partial t} = \frac{\partial^2 u}{\partial x^2}
\end{equation*}
$$

**Αρχική συνθήκη:**

- $u = 2x, \;$ για $\; 0 \leq x < 0.5$

- $u = 2(1-x), \;$ για $\; 0.5 \leq x \leq 1$

**Οριακές συνθήκες:**

Σταθερή θερμοκρασία στα δύο άκρα της

- $u = 0,  \;$ για $\; x=0$

- $u = 0,  \;$ για $\; x=1$


**Διακριτοποίηση αξόνων:**

- Χωρικό βήμα: $\;  δx = h = 0.1$

- Χρονικό βήμα: $\; δt = k = 0.001$

- $r = \frac{k}{h^2} = 0.1$


#### Explicit μέθοδος επίλυσης

#### Forward Time Centered Space (FTCS)

Χρησιμοποιούμε:

- Forward-difference ως προς το $t$

- Centered-difference ως προς το $x$

$$
\begin{align*}
& \quad \frac{\partial u}{\partial t}
=
\frac{\partial^2u}{\partial x^2}
\\[15pt]
& \Rightarrow
\frac{u_{i,j+1}-u_{i,j}}{k}
=
\frac{u_{i-1,j}-2u_{i,j}+u_{i+1,j}}{h^2}
\\[15pt]
& \Rightarrow
u_{i,j+1}-u_{i,j}
=
ru_{i-1,j}
-2ru_{i,j}
+ru_{i+1,j},
\qquad
r=\frac{k}{h^2}
\\[15pt]
& \Rightarrow
u_{i,j+1}
=
ru_{i-1,j}
+
(1-2r)u_{i,j}
+
ru_{i+1,j}
\end{align*}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


plt.style.use("default")
plt.rcParams["figure.figsize"] = [5, 2.5]  # [width_inches, height_inches]
plt.rcParams["animation.html"] = "jshtml"

#### Δημιουργήστε τον x-άξονα

$0 \leq x \leq 1$

$δx = h = 0.1$

Χρησιμοποιήστε τη συνάρτηση `np.arange`.

    np.arange(start, stop, step)

> Θυμηθείτε: stop is exclusive (δηλαδή το στοιχείο σε αυτή τη θέση δεν συμπεριλαμβάνεται).

In [ ]:
L = 1
h = 0.1
x = np.arange(0, L + h/2, h)
Nx = len(x)
x

#### Δημιουργήστε τον t-άξονα

$\; δt = k = 0.001$

Χρησιμοποιήστε τη συνάρτηση `np.arange`.

Η τελευταία χρονική στιγμή να είναι τα $0.15 \; s$.

In [ ]:
T = 0.15
k = 0.001
t = np.arange(0, T + k/2, k)
Nt = len(t)
t

#### Δημιουργία κενού πίνακα

Χρησιμοποιήστε κατάλληλα τη συνάρτηση `np.full()` για να δημιουργήσετε έναν **κενό** πίνακα `u` (δηλαδή με τιμές NaN, `np.nan`) για την αριθμητική λύση.

Η πρώτη διάσταση (δηλαδή οι γραμμές) να είναι ο χώρος και η δεύτερη διάσταση (δηλαδή οι στήλες) να είναι ο χρόνος.


In [ ]:
u = np.full((Nx, Nt), np.nan)
print(u.shape)

#### Υπενθύμιση: Boolean indexing

Επιλογή μέρους των στοιχείων ενός πίνακα με βάση μια λογική συνθήκη.

In [ ]:
a = np.array([3, 5, 4])
b = np.array([10, 2, 9])

In [ ]:
a[b > 2]

#### Αρχικές συνθήκες

- $u = 2x, \;$ για $\; 0 \leq x < 0.5$

- $u = 2(1-x), \;$ για $\; 0.5 \leq x \leq 1$

#### Αρχική συνθήκη για το πρώτο μισό της ράβδου

In [ ]:
u[x < 0.5, 0] = 2 * x[x < 0.5]


#### Αρχική συνθήκη για το δεύτερο μισό της ράβδου

In [ ]:
u[x >= 0.5, 0] = 2 * (1 - x[x >= 0.5])

#### Οριακές συνθήκες

- $u = 0,  \;$ για $\; x=0\;$ και για κάθε $t$.

- $u = 0,  \;$ για $\; x=1\;$ και για κάθε $t$.

Εισάγετε τις οριακές συνθήκες στον πίνακα `u`.

> Επιλέξτε τις κατάλληλες γραμμές του πίνακα (διάσταση χώρου) και όλες τις στήλες (διάσταση χρόνου).

In [ ]:
u[0, :] = 0
u[-1, :] = 0

#### Σχεδιάζουμε την αρχική κατάσταση της θερμοκρασίας


In [ ]:
plt.plot(x, u[:, 0])

#### Υπολογίστε τον συντελεστή $r$

$$
\begin{equation*}
r=\frac{k}{h^2}
\end{equation*}
$$

In [ ]:
r = k/h**2
print(f"r = {r:.3f}")
assert r <= 0.5, "FTCS is unstable"

#### Αριθμητική λύση της μερικής διαφορικής εξίσωσης

$$
\begin{equation*}
u^{n+1}_i = ru^n_{i-1} + (1 - 2r)u^n_i + ru^n_{i+1}
\end{equation*}
$$

#### Γράψτε την αριθμητική λύση του προβλήματος

In [ ]:
for j in range(Nt - 1):
    u[1:-1, j+1] = u[1:-1, j] + r*(
        u[:-2, j] - 2*u[1:-1, j] + u[2:, j])

Γιατί αφαιρούμε τη μονάδα στο `range(len(t) - 1)`;

Απάντηση: Υπάρχουν $N_t$ χρονικές στιγμές αλλά $N_t-1$ χρονικά βήματα. Ο δείκτης $j$ φτάνει μέχρι $N_t-2$, ώστε ο δείκτης $j+1$ να μην υπερβεί το $N_t-1$.

#### Σχεδιάστε την τελική κατάσταση της θερμοκρασίας

In [ ]:
plt.plot(x, u[:, -1], "o-")
plt.xlabel("x")
plt.ylabel("Temperature")
plt.title(f"t = {t[-1]:.3f} s")
plt.show()

#### Αναλυτική λύση της μερικής διαφορικής εξίσωσης

$$
\begin{equation*}
U = \frac{8}{\pi^2} \sum_{n=1}^{\infty}\frac{1}{n^2}\sin(\frac{1}{2}n\pi)\sin(n\pi x)e^{-n^2\pi^2t}
\end{equation*}
$$

In [ ]:
xx, tt = np.meshgrid(x, t, indexing="ij")
nterms = 100

terms = []
for n in range(1, nterms + 1):
    terms.append(
        (1/n**2) *
        np.sin(1/2 * n * np.pi) *
        np.sin(n * np.pi * xx) *
        np.exp(-n**2 * np.pi**2 * tt)
    )

U = (8/np.pi**2) * np.sum(terms, axis=0)

#### Ελέγξτε τα αποτελέσματά σας μέσω των δύο παρακάτω animation

#### 2-D animation της χρονικής εξέλιξης της θερμοκρασίας

In [ ]:
fig, ax = plt.subplots()
plt.close()


def animate(i):
    ax.clear()
    ax.plot(x, U[:, i], color="orange", label="Analytical")
    ax.scatter(x, u[:, i], color="dimgrey", label="Numerical")
    ax.set_ylim([-0.05, 1 + 0.05])
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("Temperature", fontsize=12)
    ax.set_title(f"Time: {t[i]:.3f} seconds", fontweight="bold", loc="center")
    ax.legend(loc="upper right", frameon=False)

ani1 = FuncAnimation(
    fig=fig,
    func=animate,
    frames=len(t),
    interval=25000*k,
    repeat=False,
)
ani1

#### 3-D animation της χρονικής εξέλιξης της θερμοκρασίας


In [ ]:
fig = plt.figure(figsize=(6, 4.5))
ax = plt.axes(projection="3d")
plt.close()

def animate(i):
    fig.suptitle(f"Time: {t[i]:.3f} seconds", fontsize=14, fontweight="bold")
    ax.plot(x, tt[:, i], u[:, i], color="dimgray")
    ax.set_xlim([0, x[-1]])
    ax.set_ylim([0, t[-1]])
    ax.set_zlim([0, 1])
    ax.set_xlabel("x", fontsize=16, fontweight="bold")
    ax.set_ylabel("t", fontsize=16, fontweight="bold")
    ax.set_zlabel("T", fontsize=16, fontweight="bold")
    ax.tick_params(axis="both", which="major", pad=-2.0)
    ax.view_init(azim=45)

ani2 = FuncAnimation(
    fig=fig,
    func=animate,
    frames=len(t),
    interval=25000*k,
    repeat=False,
)
plt.close()
ani2

Δοκιμάστε να επαναλάβετε το σύνολο της διαδικασίας με μεγαλύτερα χρονικά βήματα: 0.003, 0.005 και 0.008


### Μελέτη ευστάθειας

Η συνθήκη ευστάθειας είναι $r\leq 1/2$. Για $k=0.003,0.005,0.008$ προκύπτουν αντίστοιχα $r=0.3,0.5,0.8$. Η τελευταία περίπτωση είναι ασταθής. Για $k=0.008$ το $T=0.15$ δεν είναι ακέραιο πολλαπλάσιο του βήματος, οπότε χρησιμοποιούμε την τελευταία χρονική στιγμή πριν από το $T$ (0.144 s).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for dt in [0.001, 0.003, 0.005, 0.008]:
    ts = np.arange(int(np.floor(T/dt + 1e-10)) + 1)*dt
    rs = dt/h**2
    v = np.full((Nx, len(ts)), np.nan)
    v[x < 0.5, 0] = 2*x[x < 0.5]
    v[x >= 0.5, 0] = 2*(1-x[x >= 0.5])
    v[0, :] = v[-1, :] = 0
    for j in range(len(ts)-1):
        v[1:-1, j+1] = v[1:-1, j] + rs*(
            v[:-2, j] - 2*v[1:-1, j] + v[2:, j])
    ax.plot(x, v[:, -1], 'o-', label=f'k={dt}, r={rs:.1f}, t={ts[-1]:.3f}')
ax.set_xlabel('x')
ax.set_ylabel('Temperature')
ax.legend()
plt.show()